In [ ]:
# ── Environment Setup ─────────────────────────────────────────
# If running in Google Colab, upload smb_analysis.db when prompted.
# If running locally, ensure smb_analysis.db is in the working directory.

import os

try:
    from google.colab import files
    if not os.path.exists('smb_analysis.db'):
        print("Upload smb_analysis.db:")
        uploaded = files.upload()
except ImportError:
    pass  # Not in Colab

os.makedirs('charts', exist_ok=True)

if os.path.exists('smb_analysis.db'):
    print("✓ Database found")
else:
    raise FileNotFoundError("smb_analysis.db not found. Place it in the working directory.")

%% [markdown]
# SMB Growth Strategy Analysis
### Portfolio Prioritization Framework for U.S. Small and Medium-Sized Businesses

This analysis segments 34.8 million U.S. small businesses across 19 NAICS industries 
to identify high-value market segments and quantify advertising acquisition efficiency.

**Data Sources:**
- U.S. Census Bureau, Statistics of U.S. Businesses (SUSB) 2022
- SBA Office of Advocacy, 2024 Small Business Profile
- WordStream, Google Ads Industry Benchmarks 2025
- Alphabet Inc. 10-K Filings (2019–2024)

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

COLORS = {
    'blue': '#1a73e8', 'green': '#34a853', 'yellow': '#fbbc04', 'red': '#ea4335',
    'dark': '#202124', 'subtle': '#5f6368', 'light': '#f8f9fa', 'border': '#dadce0',
}
QUADRANT_COLORS = {
    'Core': COLORS['blue'], 'Growth': COLORS['green'],
    'Scale': COLORS['yellow'], 'Niche': COLORS['red'],
}

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': COLORS['border'], 'axes.labelcolor': COLORS['dark'],
    'text.color': COLORS['dark'], 'xtick.color': COLORS['subtle'],
    'ytick.color': COLORS['subtle'], 'font.family': 'sans-serif',
    'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150,
})

os.makedirs('charts', exist_ok=True)

In [ ]:
# ── Data Loading ──────────────────────────────────────────────

DB_PATH = 'smb_analysis.db'
conn = sqlite3.connect(DB_PATH)

df_landscape = pd.read_sql("SELECT * FROM smb_landscape", conn)
df_ads = pd.read_sql("SELECT * FROM google_ads_benchmarks", conn)
df_portfolio = pd.read_sql("SELECT * FROM portfolio_analysis", conn)
df_states = pd.read_sql("SELECT * FROM state_overview", conn)
df_alpha = pd.read_sql("SELECT * FROM alphabet_financials", conn)

df_landscape['payroll_per_firm'] = (
    df_landscape['payroll_small_thousands'] * 1000
    / df_landscape['employer_firms_small'].replace(0, np.nan)
)
df_landscape['avg_emp_per_firm'] = (
    df_landscape['employees_at_small_biz']
    / df_landscape['employer_firms_small'].replace(0, np.nan)
)
df_landscape['payroll_billions'] = df_landscape['payroll_small_thousands'] / 1e6

total_smb = df_landscape['total_small_businesses'].sum()
total_employer = df_landscape['employer_firms_small'].sum()
total_employees = df_landscape['employees_at_small_biz'].sum()

print(f"Industries loaded:    {len(df_landscape)}")
print(f"Total SMBs:           {total_smb:>12,.0f}")
print(f"Employer firms:       {total_employer:>12,.0f}")
print(f"SMB employees:        {total_employees:>12,.0f}")

---
## Module 1: Market Segmentation

Objective: Map the structure of the U.S. small business economy by industry,
firm size, and employment concentration.

In [ ]:
# ── 1A: Industry Distribution & Concentration Curve ──────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

df_sorted = df_landscape.sort_values('total_small_businesses', ascending=True)
labels = df_sorted['industry'].apply(lambda x: x[:35] + '...' if len(x) > 35 else x)

ax1.barh(range(len(df_sorted)), df_sorted['total_small_businesses'],
         color=COLORS['blue'], alpha=0.85, height=0.7)
ax1.set_yticks(range(len(df_sorted)))
ax1.set_yticklabels(labels, fontsize=9)
ax1.set_xlabel('Number of Small Businesses')
ax1.set_title('U.S. Small Business Count by Industry', fontsize=14, fontweight='bold', pad=15)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

for i, (_, row) in enumerate(df_sorted.tail(3).iterrows()):
    pos = len(df_sorted) - 3 + i
    ax1.annotate(f'{row["total_small_businesses"]/1e6:.1f}M',
                 xy=(row['total_small_businesses'], pos), xytext=(10, 0),
                 textcoords='offset points', fontsize=9, fontweight='bold', color=COLORS['blue'])

df_conc = df_landscape.sort_values('total_small_businesses', ascending=False).reset_index(drop=True)
df_conc['cumulative_pct'] = (
    df_conc['total_small_businesses'].cumsum()
    / df_conc['total_small_businesses'].sum() * 100
)

ax2.plot(range(1, len(df_conc)+1), df_conc['cumulative_pct'],
         'o-', color=COLORS['blue'], linewidth=2.5, markersize=6)
ax2.fill_between(range(1, len(df_conc)+1), df_conc['cumulative_pct'],
                 alpha=0.1, color=COLORS['blue'])
ax2.axhline(y=80, color=COLORS['red'], linestyle='--', alpha=0.6, label='80% threshold')
ax2.axhline(y=50, color=COLORS['yellow'], linestyle='--', alpha=0.6, label='50% threshold')

idx_50 = np.searchsorted(df_conc['cumulative_pct'].values, 50)
idx_80 = np.searchsorted(df_conc['cumulative_pct'].values, 80)
ax2.annotate(f'Top {idx_50+1} industries = 50%',
             xy=(idx_50+1, 50), xytext=(idx_50+3, 43),
             arrowprops=dict(arrowstyle='->', color=COLORS['yellow']),
             fontsize=10, color=COLORS['yellow'], fontweight='bold')
ax2.annotate(f'Top {idx_80+1} industries = 80%',
             xy=(idx_80+1, 80), xytext=(idx_80+2, 72),
             arrowprops=dict(arrowstyle='->', color=COLORS['red']),
             fontsize=10, color=COLORS['red'], fontweight='bold')

ax2.set_xlabel('Number of Industries (ranked by SMB count)')
ax2.set_ylabel('Cumulative % of Total SMBs')
ax2.set_title('Market Concentration Curve', fontsize=14, fontweight='bold', pad=15)
ax2.legend(loc='lower right')
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('charts/01_smb_landscape.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"\nConcentration: Top {idx_50+1} industries account for 50% of all SMBs.")
print(f"              Top {idx_80+1} industries account for 80%.")

In [ ]:
# ── 1B: Firm Size Distribution — Employer vs Non-Employer ────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

df_size = df_landscape.sort_values('total_small_businesses', ascending=False).head(12).copy()
short = df_size['industry'].apply(lambda x: x[:25] + '...' if len(x) > 25 else x)

ax1.barh(range(len(df_size)), df_size['firms_no_employees'],
         color=COLORS['border'], label='Non-employer', height=0.7)
ax1.barh(range(len(df_size)), df_size['firms_1_to_19'],
         left=df_size['firms_no_employees'],
         color=COLORS['blue'], label='1–19 employees', height=0.7)
ax1.barh(range(len(df_size)), df_size['firms_20_to_499'],
         left=df_size['firms_no_employees'] + df_size['firms_1_to_19'],
         color=COLORS['green'], label='20–499 employees', height=0.7)

ax1.set_yticks(range(len(df_size)))
ax1.set_yticklabels(short, fontsize=9)
ax1.set_xlabel('Number of Firms')
ax1.set_title('Firm Size Distribution by Industry', fontsize=14, fontweight='bold', pad=15)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax1.legend(loc='lower right', fontsize=9)

df_emp = df_landscape.copy()
df_emp['employer_rate'] = df_emp['employer_firms_small'] / df_emp['total_small_businesses'] * 100
df_emp = df_emp.sort_values('employer_rate', ascending=True)
short2 = df_emp['industry'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)

colors_emp = [
    COLORS['green'] if r > 30 else COLORS['yellow'] if r > 15 else COLORS['red']
    for r in df_emp['employer_rate']
]
ax2.barh(range(len(df_emp)), df_emp['employer_rate'],
         color=colors_emp, height=0.7, alpha=0.85)
ax2.set_yticks(range(len(df_emp)))
ax2.set_yticklabels(short2, fontsize=9)
ax2.set_xlabel('Employer Rate (%)')
ax2.set_title('Share of Firms with Paid Employees', fontsize=14, fontweight='bold', pad=15)
ax2.axvline(x=20, color=COLORS['subtle'], linestyle='--', alpha=0.5)

for i, (_, row) in enumerate(df_emp.iterrows()):
    ax2.annotate(f'{row["employer_rate"]:.1f}%',
                 xy=(row['employer_rate'], i), xytext=(5, 0),
                 textcoords='offset points', fontsize=8, va='center')

plt.tight_layout()
plt.savefig('charts/02_size_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"\nHighest employer rate: Accommodation & Food ({df_emp.iloc[-1]['employer_rate']:.1f}%)")
print(f"Lowest employer rate:  {df_emp.iloc[0]['industry'][:40]} ({df_emp.iloc[0]['employer_rate']:.1f}%)")
print(f"\nTarget addressable market: {total_employer:,.0f} employer firms (not {total_smb:,.0f} total)")

---
## Module 2: Attractiveness Scoring

Construct a composite attractiveness index to rank industries by their strategic
value as target segments.

In [ ]:
# ── 2A: Composite Attractiveness Ranking ─────────────────────

df_attr = df_landscape[df_landscape['employer_firms_small'] > 0].copy()

df_attr['volume_score'] = df_attr['total_small_businesses'] / df_attr['total_small_businesses'].max() * 100
df_attr['spending_score'] = df_attr['payroll_per_firm'] / df_attr['payroll_per_firm'].max() * 100
df_attr['dominance_score'] = df_attr['pct_total_emp_at_small'] / df_attr['pct_total_emp_at_small'].max() * 100

# Weights: 40% volume + 35% spending capacity + 25% employment dominance
df_attr['composite'] = (
    df_attr['volume_score'] * 0.4
    + df_attr['spending_score'] * 0.35
    + df_attr['dominance_score'] * 0.25
)
df_attr = df_attr.sort_values('composite', ascending=False)

fig, ax = plt.subplots(figsize=(16, 9))
short = df_attr['industry'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)
y_pos = range(len(df_attr))

ax.barh(y_pos, df_attr['volume_score'] * 0.4,
        color=COLORS['blue'], alpha=0.85, label='Volume (40%)', height=0.6)
ax.barh(y_pos, df_attr['spending_score'] * 0.35,
        left=df_attr['volume_score'] * 0.4,
        color=COLORS['green'], alpha=0.85, label='Spending Capacity (35%)', height=0.6)
ax.barh(y_pos, df_attr['dominance_score'] * 0.25,
        left=df_attr['volume_score'] * 0.4 + df_attr['spending_score'] * 0.35,
        color=COLORS['yellow'], alpha=0.85, label='Employment Dominance (25%)', height=0.6)

ax.set_yticks(y_pos)
ax.set_yticklabels(short, fontsize=9)
ax.set_xlabel('Composite Attractiveness Score (0–100)')
ax.set_title('Industry Attractiveness Ranking', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='lower right')

for i, (_, row) in enumerate(df_attr.iterrows()):
    ax.annotate(f'{row["composite"]:.0f}',
                xy=(row['composite'], i), xytext=(8, 0),
                textcoords='offset points', fontsize=9, fontweight='bold', color=COLORS['dark'])

plt.tight_layout()
plt.savefig('charts/03_attractiveness.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nTop 5 industries by composite attractiveness:")
for i, (_, row) in enumerate(df_attr.head(5).iterrows()):
    print(f"  {i+1}. {row['industry'][:50]:50s}  Score: {row['composite']:.1f}")

In [ ]:
# ── 2B: Portfolio Matrix — Volume × Spending Capacity ────────

fig, ax = plt.subplots(figsize=(14, 9))

df_bubble = df_landscape[df_landscape['employer_firms_small'] > 100].copy()
df_bubble = df_bubble.merge(
    df_portfolio[['industry', 'priority_quadrant']].drop_duplicates(),
    on='industry', how='left'
)
df_bubble['priority_quadrant'] = df_bubble['priority_quadrant'].fillna('Niche')

for quadrant, color in QUADRANT_COLORS.items():
    mask = df_bubble['priority_quadrant'] == quadrant
    subset = df_bubble[mask]
    ax.scatter(
        subset['total_small_businesses'], subset['payroll_per_firm'],
        s=subset['employees_at_small_biz'] / 15000, alpha=0.65,
        color=color, edgecolors='white', linewidth=1.5, label=quadrant, zorder=5
    )

for _, row in df_bubble.iterrows():
    if row['total_small_businesses'] > 2e6 or row['payroll_per_firm'] > 500000:
        ax.annotate(row['industry'][:25],
                    (row['total_small_businesses'], row['payroll_per_firm']),
                    fontsize=8, ha='center', va='bottom',
                    xytext=(0, 10), textcoords='offset points')

ax.axhline(y=300000, color=COLORS['subtle'], linestyle='--', alpha=0.4)
ax.axvline(x=1000000, color=COLORS['subtle'], linestyle='--', alpha=0.4)

ax.text(3.5e6, 650000, 'CORE', fontsize=11, fontweight='bold', color=COLORS['blue'], alpha=0.5, ha='center')
ax.text(4e5, 650000, 'GROWTH', fontsize=11, fontweight='bold', color=COLORS['green'], alpha=0.5, ha='center')
ax.text(3.5e6, 150000, 'SCALE', fontsize=11, fontweight='bold', color=COLORS['yellow'], alpha=0.5, ha='center')
ax.text(4e5, 150000, 'NICHE', fontsize=11, fontweight='bold', color=COLORS['red'], alpha=0.5, ha='center')

ax.set_xlabel('Total Small Businesses', fontsize=12)
ax.set_ylabel('Average Payroll per Firm ($)', fontsize=12)
ax.set_title('Portfolio Priority Matrix: Volume × Spending Capacity',
             fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.legend(title='Quadrant', loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.savefig('charts/04_portfolio_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

core_count = (df_bubble['priority_quadrant'] == 'Core').sum()
print(f"\n{core_count} industries in the Core quadrant (high volume + high spending).")

---
## Module 3: Acquisition Efficiency

Overlay Google Ads cost benchmarks onto the SMB landscape to identify
segments with the best return on acquisition spend.

In [ ]:
# ── 3A: CPC Distribution & Leads per $1,000 ─────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

df_cpc = df_ads.sort_values('avg_cpc_usd', ascending=True)
short_cpc = df_cpc['industry'].apply(lambda x: x[:22] + '...' if len(x) > 22 else x)
colors_cpc = [
    COLORS['green'] if c < 3 else COLORS['yellow'] if c < 5 else COLORS['red']
    for c in df_cpc['avg_cpc_usd']
]

ax1.barh(range(len(df_cpc)), df_cpc['avg_cpc_usd'],
         color=colors_cpc, height=0.7, alpha=0.85)
ax1.set_yticks(range(len(df_cpc)))
ax1.set_yticklabels(short_cpc, fontsize=9)
ax1.set_xlabel('Average CPC ($)')
ax1.set_title('Google Ads CPC by Industry (2025)', fontsize=14, fontweight='bold', pad=15)

for i, (_, row) in enumerate(df_cpc.iterrows()):
    ax1.annotate(f'${row["avg_cpc_usd"]:.2f}',
                 xy=(row['avg_cpc_usd'], i), xytext=(5, 0),
                 textcoords='offset points', fontsize=8, va='center')

df_leads = df_ads.copy()
df_leads['leads_per_1k'] = 1000 / df_leads['avg_cpl_usd']
df_leads = df_leads.sort_values('leads_per_1k', ascending=True)
short_leads = df_leads['industry'].apply(lambda x: x[:22] + '...' if len(x) > 22 else x)

colors_leads = [
    COLORS['green'] if l > 20 else COLORS['yellow'] if l > 10 else COLORS['red']
    for l in df_leads['leads_per_1k']
]
ax2.barh(range(len(df_leads)), df_leads['leads_per_1k'],
         color=colors_leads, height=0.7, alpha=0.85)
ax2.set_yticks(range(len(df_leads)))
ax2.set_yticklabels(short_leads, fontsize=9)
ax2.set_xlabel('Leads per $1,000 Ad Spend')
ax2.set_title('Lead Generation Efficiency by Industry', fontsize=14, fontweight='bold', pad=15)

for i, (_, row) in enumerate(df_leads.iterrows()):
    ax2.annotate(f'{row["leads_per_1k"]:.1f}',
                 xy=(row['leads_per_1k'], i), xytext=(5, 0),
                 textcoords='offset points', fontsize=8, va='center')

plt.tight_layout()
plt.savefig('charts/05_ads_efficiency.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 3B: CPC vs Conversion Rate Scatter ──────────────────────

fig, ax = plt.subplots(figsize=(12, 8))

bubble_size = 1000 / df_ads['avg_cpl_usd'] * 15
colors_scatter = [
    COLORS['green'] if c < 3 else COLORS['yellow'] if c < 5 else COLORS['red']
    for c in df_ads['avg_cpc_usd']
]

ax.scatter(df_ads['avg_cpc_usd'], df_ads['avg_cvr_pct'],
           s=bubble_size, alpha=0.65, c=colors_scatter,
           edgecolors='white', linewidth=1.5, zorder=5)

for _, row in df_ads.iterrows():
    ax.annotate(row['industry'][:18],
                (row['avg_cpc_usd'], row['avg_cvr_pct']),
                fontsize=7, ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points')

ax.axhspan(8, 15, xmin=0, xmax=0.35, alpha=0.06, color=COLORS['green'])
ax.text(1.2, 14, 'Optimal zone:\nlow cost, high conversion',
        fontsize=9, color=COLORS['green'], fontweight='bold', alpha=0.7)

ax.set_xlabel('Average Cost per Click ($)', fontsize=12)
ax.set_ylabel('Average Conversion Rate (%)', fontsize=12)
ax.set_title('CPC vs Conversion Rate by Industry',
             fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('charts/06_cpc_vs_cvr.png', bbox_inches='tight', dpi=150)
plt.show()

sweet_spot = df_ads[(df_ads['avg_cpc_usd'] < 4) & (df_ads['avg_cvr_pct'] > 8)]
print("High-efficiency verticals (CPC < $4 and CVR > 8%):")
for _, row in sweet_spot.iterrows():
    print(f"  {row['industry']:35s}  CPC: ${row['avg_cpc_usd']:.2f}  CVR: {row['avg_cvr_pct']:.1f}%")

---
## Module 4: Platform Revenue Context

Quantify the relationship between SMB advertising and platform-level economics.

In [ ]:
# ── 4A: Alphabet Revenue Breakdown ───────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

df_alpha['total_ads'] = (
    df_alpha['google_search_ads_b']
    + df_alpha['google_network_ads_b']
    + df_alpha['youtube_ads_b']
)
df_alpha['non_ads'] = df_alpha['total_revenue_b'] - df_alpha['total_ads']

ax1.bar(df_alpha['year'], df_alpha['google_search_ads_b'],
        color=COLORS['blue'], label='Search Ads', alpha=0.85)
ax1.bar(df_alpha['year'], df_alpha['google_network_ads_b'],
        bottom=df_alpha['google_search_ads_b'],
        color=COLORS['green'], label='Network Ads', alpha=0.85)
ax1.bar(df_alpha['year'], df_alpha['youtube_ads_b'],
        bottom=df_alpha['google_search_ads_b'] + df_alpha['google_network_ads_b'],
        color=COLORS['red'], label='YouTube Ads', alpha=0.85)
ax1.bar(df_alpha['year'], df_alpha['non_ads'],
        bottom=df_alpha['total_ads'],
        color=COLORS['border'], label='Cloud & Other', alpha=0.85)

ax1.set_ylabel('Revenue ($B)')
ax1.set_title('Alphabet Revenue Composition', fontsize=14, fontweight='bold', pad=15)
ax1.legend(loc='upper left', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))

ads_pct = df_alpha['total_ads'] / df_alpha['total_revenue_b'] * 100
ax2.plot(df_alpha['year'], ads_pct, 'o-',
         color=COLORS['blue'], linewidth=2.5, markersize=8)
ax2.fill_between(df_alpha['year'], ads_pct, alpha=0.1, color=COLORS['blue'])
ax2.set_ylabel('Advertising as % of Total Revenue')
ax2.set_title('Advertising Revenue Share', fontsize=14, fontweight='bold', pad=15)
ax2.set_ylim(60, 90)
ax2.axhline(y=75, color=COLORS['subtle'], linestyle='--', alpha=0.4)

for year, pct in zip(df_alpha['year'], ads_pct):
    ax2.annotate(f'{pct:.1f}%', (year, pct),
                 textcoords='offset points', xytext=(0, 10),
                 fontsize=9, fontweight='bold', ha='center')

plt.tight_layout()
plt.savefig('charts/07_alphabet_revenue.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"\n2024: ${df_alpha.iloc[-1]['total_ads']:.0f}B in advertising revenue "
      f"({ads_pct.iloc[-1]:.1f}% of total)")
print(f"Search Ads alone: ${df_alpha.iloc[-1]['google_search_ads_b']:.0f}B "
      f"(2x growth from 2019)")

In [ ]:
# ── 4B: Combined Opportunity Map ─────────────────────────────
# Final synthesis: industries plotted by volume, spending capacity,
# and acquisition cost (bubble size ∝ 1/CPC).

fig, ax = plt.subplots(figsize=(14, 10))

df_opp = df_portfolio[df_portfolio['avg_cpc_usd'].notna()].copy()

for quadrant, color in QUADRANT_COLORS.items():
    mask = df_opp['priority_quadrant'] == quadrant
    subset = df_opp[mask]
    sizes = (10 / subset['avg_cpc_usd']) ** 2 * 100
    ax.scatter(
        subset['total_small_businesses'], subset['avg_payroll_per_firm'],
        s=sizes, alpha=0.6, color=color,
        edgecolors='white', linewidth=2, label=quadrant, zorder=5
    )

for _, row in df_opp.iterrows():
    name = row['industry'][:22]
    cpc = row['avg_cpc_usd']
    ax.annotate(
        f'{name}\nCPC: ${cpc:.2f}',
        (row['total_small_businesses'], row['avg_payroll_per_firm']),
        fontsize=7, ha='center', va='bottom',
        xytext=(0, 12), textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                  edgecolor=COLORS['border'], alpha=0.8)
    )

ax.axhline(y=300000, color=COLORS['subtle'], linestyle='--', alpha=0.3)
ax.axvline(x=1000000, color=COLORS['subtle'], linestyle='--', alpha=0.3)

ax.set_xlabel('Total Small Businesses', fontsize=12)
ax.set_ylabel('Average Payroll per Firm ($)', fontsize=12)
ax.set_title('Strategic Opportunity Map\n(Bubble size ∝ 1/CPC — larger = lower acquisition cost)',
             fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.legend(title='Priority Quadrant', loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.savefig('charts/08_opportunity_map.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Summary

| Finding | Detail |
|---------|--------|
| Market concentration | Top 5 industries = 56% of 34.8M SMBs |
| Core targets | 9 industries with high volume + high spending capacity |
| Best acquisition ROI | Retail Trade: 3.0M firms, $355K payroll/firm, $1.63 CPC |
| Efficiency optimization | Construction & Prof. Services: high value, high CPC |
| Platform dependency | Ads = 76% of Alphabet revenue ($268B in 2024) |

In [ ]:
# ── Final metrics table ──────────────────────────────────────

print("=" * 90)
print("PORTFOLIO PRIORITIZATION — SUMMARY METRICS")
print("=" * 90)
print(f"\n{'Industry':<45} {'SMBs':>10} {'$/Firm':>10} {'CPC':>7} {'Quadrant':>10}")
print("-" * 90)

for _, row in df_opp.sort_values('total_small_businesses', ascending=False).iterrows():
    print(f"{row['industry'][:44]:<45} "
          f"{row['total_small_businesses']:>10,.0f} "
          f"${row['avg_payroll_per_firm']:>8,.0f} "
          f"${row['avg_cpc_usd']:>5.2f} "
          f"{row['priority_quadrant']:>10}")

conn.close()